# 3. Two backend models behind one alias

Configure a model alias with two compatible Chat Completions targets at equal priority and weight; see **Two-model pool demo** in the repository README. Use the same three settings as the other notebooks in the root `.env`: `AI_GATEWAY_CHAT_URL`, `AI_GATEWAY_API_KEY`, and `AI_GATEWAY_MODEL_ALIAS`. Set the shared model alias to your pooled alias; no separate pool setting is needed. Shell environment variables take precedence. This notebook always calls that **one pooled alias**. It sends at most 12 short sequential requests; a 1:1 weight is a *long-run preference*, not a guarantee of six answers from each backend.

**Demo:** Show the alias targets in the portal, run the cells, and point out the reported model IDs and counts. Use per-backend telemetry to verify routing; wording and reported IDs alone are not definitive evidence.

In [ ]:
import os
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv

lab_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(lab_root / '.env', override=False)

chat_url = os.getenv('AI_GATEWAY_CHAT_URL', '').strip()
api_key = os.getenv('AI_GATEWAY_API_KEY', '').strip()
alias = os.getenv('AI_GATEWAY_MODEL_ALIAS', '').strip()
if not all((chat_url, api_key, alias)):
    raise RuntimeError('Set AI_GATEWAY_CHAT_URL, AI_GATEWAY_API_KEY, and AI_GATEWAY_MODEL_ALIAS in the root .env or shell environment.')
parsed = urlparse(chat_url)
if parsed.scheme != 'https' or not parsed.netloc or not parsed.path.endswith('/chat/completions'):
    raise ValueError('Use the full HTTPS Chat completions endpoint copied from AI Gateway.')
print('Gateway:', parsed.hostname, '| pooled alias:', alias)


## Compare individual answers

Each request has the **same URL, alias, and prompt**. The `model` field, if returned, is provider-reported; it may identify the actual backend, show the alias, or be absent. Distinct answers demonstrate variation but *do not on their own establish* that requests reached different backends.

In [ ]:
observations = []
for number in range(1, 13):
    response = requests.post(
        chat_url,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={'model': alias, 'messages': [
            {'role': 'user', 'content': 'Give one short, original analogy for an AI gateway.'}
        ]},
        timeout=90,
    )
    if response.status_code == 429:
        print('Stopped at request', number, 'after a 429; check the gateway policy and backend quota.')
        break
    response.raise_for_status()
    result = response.json()
    answer = result['choices'][0]['message']['content']
    observations.append({'request': number, 'reported_model': result.get('model'), 'answer': answer})
    print(f"{number:02d} | model={result.get('model', 'not reported')} | {answer[:180]}")


## Interpret the evidence

The counts below refer to **reported response model IDs**, not proven backend resource IDs. If two distinguishable IDs appear, they support the routing demonstration. If one ID appears, it does not disprove pooling: both deployments might report the same model, or a small sample might select only one target. Verify selection in AI Gateway telemetry grouped by backend model, after enabling a telemetry destination. Avoid inferring routing from wording differences alone.

In [ ]:
counts = Counter(item['reported_model'] or '<not reported>' for item in observations)
print('Responses by reported model:', dict(counts))
print('Completed requests:', len(observations), '(same alias:', alias + ')')
if len(counts) < 2:
    print('Confirm both backend resource IDs, weights, and actual routing in gateway telemetry.')
